# 03 · Storage & data management

**Phase skill: designing a schema and querying it with SQL.** Raw JSON is fine for a
download and terrible for questions. You will design a table for the spells you pulled
in 02, load it, and answer three questions **with SQL only** — pandas may display the
results, but every answer must come out of a single SQL query.

Your table lives in `workbook/data/workbook.db`, your own scratch database. The main
`data/monsterlab.db` is never touched by this workbook.

In [ ]:
import json
import sqlite3
from pathlib import Path

import pandas as pd

RAW_PATH = Path("data") / "raw" / "open5e_spells.json"
assert RAW_PATH.exists(), "Run 02_acquire.ipynb first -- it saves the raw spells JSON this notebook loads."
spells = json.loads(RAW_PATH.read_text())
print(f"{len(spells)} raw spell records loaded")

conn = sqlite3.connect(Path("data") / "workbook.db")
# Rerunnable top-to-bottom: drop the workbook table so 3.1 recreates it fresh.
conn.execute("DROP TABLE IF EXISTS wb_spells")
conn.commit()

## Exercise 3.1 — design the table

Write a `CREATE TABLE` statement for a table named **`wb_spells`** and execute it. The
required columns, described in words (choosing SQL types is your job):

| column | holds |
|--------|-------|
| `slug` | the record's short unique identifier (text) |
| `name` | the spell's display name (text, must never be missing) |
| `level` | the spell level, a whole number 0–9 — the raw field is called `level_int` |
| `school` | the school of magic (text) |
| `concentration` | whether it needs concentration (the raw field holds text) |
| `range` | the listed range (text) |
| `duration` | the listed duration (text) |
| `dnd_class` | comma-separated caster classes (text) |

**Produce:** the executed statement — the check reads the schema back out of SQLite.

<details><summary>Hint 1 (concept)</summary>

A schema is a promise about the future: types say what values are legal, constraints say what must always be present. Decide what should be impossible to store.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up SQLite CREATE TABLE syntax, column constraints NOT NULL and UNIQUE, and note that `range` needs quoting care in some SQL dialects (SQLite allows it plain).

</details>

In [ ]:
# Produce: an executed CREATE TABLE for wb_spells

create_sql = ...

conn.execute(create_sql)

In [ ]:
info = conn.execute("PRAGMA table_info(wb_spells)").fetchall()
assert info, "no table named wb_spells exists yet"
cols = {row[1]: (row[2] or '').upper() for row in info}
required_text = ["slug", "name", "school", "concentration", "range", "duration", "dnd_class"]
missing = sorted((set(required_text) | {"level"}) - set(cols))
assert not missing, f"wb_spells is missing columns: {missing}"
for col in required_text:
    assert "CHAR" in cols[col] or "TEXT" in cols[col], (
        f"column {col!r} is declared {cols[col] or 'untyped'!r} -- expected a text type")
assert "INT" in cols["level"], f"column 'level' is declared {cols['level'] or 'untyped'!r} -- expected an integer type"
notnull = {row[1] for row in info if row[3]}
assert "name" in notnull, "the name column can currently be NULL -- it was required to never be missing"
print(f"PASS: wb_spells exists with all {len(cols)} columns typed sensibly.")
print("Context: PRAGMA table_info is SQLite asking a table to describe itself -- schema as data.")

## Exercise 3.2 — load the rows

Insert **every** record from `spells` into `wb_spells` using a single `executemany`,
then commit. Remember the raw field `level_int` maps to your `level` column.

**Produce:** a fully loaded `wb_spells` table.

<details><summary>Hint 1 (concept)</summary>

executemany pairs one parameterized INSERT with a sequence of value tuples -- build the sequence from the list of dicts first.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `sqlite3` parameter placeholders (`?`) and `Connection.executemany`.

</details>

In [ ]:
# Produce: all records inserted into wb_spells (one executemany), then committed

...

In [ ]:
n = conn.execute("SELECT COUNT(*) FROM wb_spells").fetchone()[0]
assert n > 0, "wb_spells is empty -- no rows were inserted"
assert n == len(spells), f"wb_spells holds {n} rows but the raw file has {len(spells)}"
lo, hi = conn.execute("SELECT MIN(level), MAX(level) FROM wb_spells").fetchone()
assert (lo, hi) == (0, 9), f"levels span {lo}-{hi}, expected 0-9 -- was level_int mapped into the level column?"
row = conn.execute("SELECT school, level FROM wb_spells WHERE name = 'Fireball'").fetchone()
assert row is not None, "Fireball is missing from wb_spells"
assert row == ("Evocation", 3), f"Fireball is stored as {row}, expected ('Evocation', 3) -- are values in the right columns?"
print(f"PASS: all {n} spells stored, spot checks line up.")
print("Context: from here on, questions cost one query instead of one loop over JSON.")

## Exercise 3.3 — counts by school

**One SQL query:** how many spells does each school of magic have? Most numerous school
first.

**Produce:** `spells_by_school` — a dataframe with exactly the columns `school` and `n`,
sorted by `n` descending.

<details><summary>Hint 1 (concept)</summary>

Counting per group is aggregation: collapse the rows that share a school into one row carrying the group's size.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up GROUP BY, COUNT(*) with a column alias, ORDER BY ... DESC, and `pd.read_sql`.

</details>

In [ ]:
# Produce: spells_by_school (DataFrame: school, n) -- one SQL query

spells_by_school = ...

spells_by_school

In [ ]:
assert list(spells_by_school.columns) == ["school", "n"], (
    f"columns are {list(spells_by_school.columns)}, expected ['school', 'n']")
assert 6 <= len(spells_by_school) <= 10, f"{len(spells_by_school)} schools found -- expected about 8"
assert spells_by_school["n"].is_monotonic_decreasing, "rows are not sorted by count, largest first"
top = spells_by_school.iloc[0]
assert top["school"] == "Evocation", f"the most numerous school came out as {top['school']!r}"
assert 54 <= top["n"] <= 66, f"Evocation shows {top['n']} spells -- expected about 60"
print(f"PASS: {len(spells_by_school)} schools, Evocation on top with {top['n']}.")
print("Context: a GROUP BY count is usually the first honest look at any categorical column.")

## Exercise 3.4 — filter and order

**One SQL query:** every **level 3 Evocation** spell, alphabetical by name, returning
the columns `name` and `range`.

**Produce:** `lv3_evocation` — that dataframe.

<details><summary>Hint 1 (concept)</summary>

Filtering rows and ordering results are separate clauses doing separate jobs -- conditions first, presentation last.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up WHERE with AND, and ORDER BY.

</details>

In [ ]:
# Produce: lv3_evocation (DataFrame: name, range) -- one SQL query

lv3_evocation = ...

lv3_evocation

In [ ]:
assert list(lv3_evocation.columns) == ["name", "range"], (
    f"columns are {list(lv3_evocation.columns)}, expected ['name', 'range']")
names = list(lv3_evocation["name"])
assert names == sorted(names), "names are not in alphabetical order"
assert 5 <= len(names) <= 9, f"{len(names)} rows -- expected about 7 level-3 Evocation spells"
for expected_name in ("Daylight", "Fireball", "Lightning Bolt"):
    assert expected_name in names, f"{expected_name} is missing -- check both filter conditions"
print(f"PASS: {len(names)} level-3 Evocation spells, Daylight first.")
print("Context: WHERE trims rows before anything is counted; notice 3.5 needs a different tool.")

## Exercise 3.5 — filter *groups*, not rows

**One SQL query:** which schools have **more than 40** spells? A `WHERE` can't answer
this — the count doesn't exist until after grouping — so this is the job of `GROUP BY`
with `HAVING`.

**Produce:** `big_schools` — a Python `set` of the qualifying school names.

<details><summary>Hint 1 (concept)</summary>

WHERE filters rows before aggregation; HAVING filters the aggregated groups after. The condition here is about a group property, so it must run after the groups exist.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up HAVING with COUNT(*), and how to turn a single query column into a set.

</details>

In [ ]:
# Produce: big_schools (set of school names with more than 40 spells) -- one SQL query

big_schools = ...

big_schools

In [ ]:
assert isinstance(big_schools, set), f"big_schools should be a set, got {type(big_schools).__name__}"
assert 2 <= len(big_schools) <= 4, f"{len(big_schools)} schools qualified -- expected 3"
for s in ("Evocation", "Transmutation"):
    assert s in big_schools, f"{s} has over 40 spells but is missing from big_schools"
assert "Necromancy" not in big_schools, "Necromancy is well under 40 spells -- is the HAVING threshold right?"
print(f"PASS: {sorted(big_schools)}.")
print("Context: WHERE/HAVING is the same split pandas makes between filtering rows and filtering groupby results.")

**Done.** You designed a schema, loaded it, and asked it three questions in its own
language. `04_clean.ipynb` goes back to the Shadowdark side of the house, where the raw
text is much less cooperative.